# LOPO Analysis

In [26]:
import pandas as pd
import json

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR,NuSVR

import pandas as pd
from helpers.modeling import (
    identify_column_types,
    create_preprocessor,
    evaluate_model,
)

from tqdm import tqdm


with open('results.json', 'r') as f:
    results = json.load(f)

In [27]:
df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv")
df.head()
df['paper_reference'].nunique()
df['paper_reference'].value_counts().describe()

count    165.000000
mean      12.563636
std       15.086454
min        1.000000
25%        4.000000
50%        8.000000
75%       16.000000
max      112.000000
Name: count, dtype: float64

In [28]:
def run_pipeline(model_cls, model_key, kernel_kwargs=None):
    params = results["best_params"]["best_params_publications_included"][model_key]
    pipeline = Pipeline([('preprocessor', preprocessor),
                          ('model', model_cls(**(kernel_kwargs or {})))])
    pipeline.set_params(**params)
    pipeline.fit(X_train, y_train)

    train_metrics = evaluate_model(y_train, pipeline.predict(X_train))
    test_metrics = evaluate_model(y_test, pipeline.predict(X_test))
    return pipeline, train_metrics, test_metrics

In [29]:
X = df.drop(columns=["cs_28d", "paper_reference"])
y = df["cs_28d"]

pub_col = "paper_reference" 

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns, 
                                   handle_unknown='ignore') 


In [30]:
total_rows = 0
for pub, group in df.groupby('paper_reference'):
    print(f"{pub}: {len(group)} rows")
    total_rows += len(group)
    
print(total_rows)

Ref-1-data: 16 rows
Ref-10-Research: 7 rows
Ref-100-Research: 8 rows
Ref-101-Research: 28 rows
Ref-102-Research: 16 rows
Ref-103-Research: 10 rows
Ref-104-Research: 8 rows
Ref-105-Research: 7 rows
Ref-106-Research: 8 rows
Ref-107-Research: 16 rows
Ref-108-Research: 3 rows
Ref-109-Research: 2 rows
Ref-11-Research : 4 rows
Ref-110-Research: 18 rows
Ref-111-Research: 2 rows
Ref-113-Research: 5 rows
Ref-114-Research: 5 rows
Ref-115-Research: 5 rows
Ref-116-Research: 36 rows
Ref-117-Research: 5 rows
Ref-118-Research: 5 rows
Ref-119-Research: 1 rows
Ref-12-Research: 4 rows
Ref-120-Research: 4 rows
Ref-121-Research: 80 rows
Ref-122-Research: 18 rows
Ref-123-Research: 11 rows
Ref-124-Research: 10 rows
Ref-125-Research: 16 rows
Ref-126-Research: 5 rows
Ref-127-Research: 3 rows
Ref-128-Research: 3 rows
Ref-129-Research: 1 rows
Ref-13-Research: 27 rows
Ref-130-Research: 6 rows
Ref-132-Research: 12 rows
Ref-133-Research: 2 rows
Ref-134-Research: 20 rows
Ref-135-Research: 43 rows
Ref-136-Research: 

In [31]:
model_configs = [
    (KNeighborsRegressor, 'knn', None),
    (SVR, 'svr', {'kernel': 'rbf'}),
    (NuSVR, 'nusvr', {'kernel': 'rbf'}),
]

groups = list(df.groupby(pub_col))
threshold = 50
lopo_results = []
lopo_predictions = []  
total_rows = 0

for pub_id, group_df in tqdm(groups, total=len(groups)):
    if len(group_df) < threshold:   #threshold
        continue
    total_rows += len(group_df)

    train_mask = df[pub_col] != pub_id
    test_mask  = df[pub_col] == pub_id

    X_train, y_train = X[train_mask], y[train_mask]
    X_test,  y_test  = X[test_mask],  y[test_mask]

    for model_cls, model_key, kwargs in model_configs:
        pipeline, train_metrics, test_metrics = run_pipeline(model_cls, model_key, kwargs)
        test_metrics.update({'publication': pub_id, 'model': model_key, 'train_RMSE': train_metrics['RMSE']})
        lopo_results.append(test_metrics)

        y_pred = pipeline.predict(X_test)
        for true_val, pred_val, idx in zip(y_test.values, y_pred, y_test.index):
            lopo_predictions.append({'index': idx, 'publication': pub_id, 'model': model_key,
                                       'y_true': true_val, 'y_pred': pred_val})

lopo_df = pd.DataFrame(lopo_results)
lopo_pred_df = pd.DataFrame(lopo_predictions)
print(f"total rows after thresholding ({threshold}) : {total_rows}")

  0%|          | 0/165 [00:00<?, ?it/s]

100%|██████████| 165/165 [00:09<00:00, 16.58it/s]

total rows after thresholding (50) : 452


In [32]:
print(f"unique values after threshold  {threshold}: {lopo_df['publication'].nunique()}")


unique values after threshold  50: 6


In [33]:
# Pooled LOPO summary:
# Every row in the dataset was held out exactly once (by the model that never
# saw its publication during training). This pools ALL those held-out
# predictions together (across all 165 eligible publications) and computes
# ONE overall RMSE/MAE/R2/Correlation/Mean_Residual per model - as if it were
# a single big "unseen publication" test set.

pooled_summary = []
for model_key, group in lopo_pred_df.groupby('model'):
    m = evaluate_model(group['y_true'], group['y_pred'])
    m['model'] = model_key
    m['N_total'] = len(group)
    pooled_summary.append(m)

pooled_lopo_df = pd.DataFrame(pooled_summary)
pooled_lopo_df

,RMSE,MAE,MaxAE,R2,Correlation,Mean_Residual,N,model,N_total
0,25.034110,19.486583,76.470573,0.405044,0.654923,4.478453,452,knn,452
1,31.771124,23.384036,105.519257,0.041735,0.420663,8.286032,452,nusvr,452
2,32.385412,24.167832,108.198741,0.004321,0.429739,9.722705,452,svr,452


In [34]:
lopo_pred_df['residual'] = lopo_pred_df['y_true'] - lopo_pred_df['y_pred']

residual_pivot = lopo_pred_df.pivot(index='index', columns='model', values='residual')
residual_pivot['mean_abs_residual'] = residual_pivot[['knn', 'svr', 'nusvr']].abs().mean(axis=1)
residual_pivot['same_direction'] = residual_pivot[['knn', 'svr', 'nusvr']].apply(
    lambda r: (r > 0).all() or (r < 0).all(), axis=1
)

worst_rows = residual_pivot.sort_values('mean_abs_residual', ascending=False).head(10)
worst_rows

model,knn,nusvr,svr,mean_abs_residual,same_direction
index,,,,,
547,29.372722,105.519257,108.198741,81.030240,True
548,27.370244,103.170239,107.099435,79.213306,True
549,22.368151,96.876887,100.749463,73.331500,True
551,41.339236,86.949285,89.136436,72.474986,True
527,31.012123,91.362238,94.050150,72.141504,True
546,22.375701,95.000376,95.113002,70.829693,True
507,47.810665,79.770832,82.650846,70.077448,True
528,28.973918,88.202073,92.418069,69.864687,True
1036,72.746268,66.282123,70.138897,69.722429,True


In [35]:
worst_idx = worst_rows.head(10).index 

df.loc[worst_idx].T

index,547,548,549,551,527,546,507,528,1036,531
cement,850.0,850.0,850.0,850.0,850.0,850.0,850.0,850.0,960.0,850.0
cement_type,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_53,OPC_42.5,OPC_53
silica_fume,260.0,260.0,260.0,260.0,260.0,260.0,260.0,260.0,288.0,260.0
fly_ash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
fly_ash_type,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable,not_applicable
limestone_powder,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
quartz_powder,212.0,212.0,212.0,212.0,212.0,212.0,212.0,212.0,0.0,212.0
glass_powder,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
rice_husk_ash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
metakaolin,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
lopo_df.to_csv("results/lopo_results.csv", index=False)

In [37]:
summary = lopo_df.groupby('model')[['RMSE', 'MAE', 'MaxAE', 'R2', 'Correlation', 'Mean_Residual']].agg(['mean', 'median', 'std'])
summary

RMSE                              MAE                        \
            mean     median        std       mean     median        std   
model                                                                     
knn    24.348190  23.682331   7.237861  19.644830  18.754991   6.001743   
nusvr  28.967939  24.510594  16.093381  23.996147  18.639374  15.289582   
svr    29.956269  24.638939  15.784106  24.972631  18.691794  14.866152   

           MaxAE                              R2                      \
            mean     median        std      mean    median       std   
model                                                                  
knn    62.127685  62.743130  12.365418 -0.020991 -0.018566  0.356292   
nusvr  65.834977  63.455309  23.423830 -0.521338 -0.172720  1.337106   
svr    66.282528  62.275742  22.877341 -0.674436 -0.527917  1.341686   

      Correlation                     Mean_Residual                       
             mean    median       std          mean    median        std  
model                                                                     
knn      0.515743  0.556080  0.253858      5.401336  3.517100  12.206427  
nusvr    0.454965  0.557807  0.419517      9.474200  0.864509  19.552964  
svr      0.466052  0.522647  0.391268     10.618003  3.643223  20.765849

In [38]:
per_pub = lopo_df[['publication', 'model', 'N', 'RMSE', 'MAE', 'MaxAE', 'R2', 'Correlation', 'Mean_Residual']].sort_values(['model', 'publication'])
per_pub.to_csv("results/lopo_per_publication.csv", index=False)

In [39]:
#worst publivations
worst = lopo_df.sort_values('R2').groupby('model').head(10)[['model', 'publication', 'N', 'RMSE', 'R2', 'Correlation', 'Mean_Residual']]
worst

,model,publication,N,RMSE,R2,Correlation,Mean_Residual
13,svr,Ref-48-Research,72,57.664809,-3.157912,-0.111112,38.178641
14,nusvr,Ref-48-Research,72,57.017799,-3.065130,-0.161898,36.340555
4,svr,Ref-139-Research,51,26.696770,-0.826831,0.180388,-11.666081
5,nusvr,Ref-139-Research,51,25.506765,-0.667600,0.084374,-6.969362
16,svr,Ref-85-Research,64,38.569044,-0.610416,0.929342,34.342083
17,nusvr,Ref-85-Research,64,37.098202,-0.489930,0.932176,32.143649
15,knn,Ref-85-Research,64,36.624182,-0.452099,0.824986,26.224895
7,svr,Ref-141-Research,73,15.012060,-0.445419,0.368221,6.219207
6,knn,Ref-141-Research,73,14.496286,-0.347804,0.294726,8.156187
3,knn,Ref-139-Research,51,21.505365,-0.185427,0.148634,-1.121987


In [40]:
lopo_df['direction'] = lopo_df['Mean_Residual'].apply(lambda x: 'under-predicted' if x > 0 else 'over-predicted')
direction_summary = lopo_df.groupby(['model', 'direction']).size().unstack(fill_value=0)
direction_summary

direction,over-predicted,under-predicted
model,,
knn,3,3
nusvr,3,3
svr,2,4
